# 🦍 Système Expert — Détection de Gorilles de Montagne
**Architecture hybride : YOLOv11n + Module de Validation Heuristique (MVH)**

| Formule | Description |
|---|---|
| `C₁ = 0.4·(L̄/255) + 0.4·R_b + 0.2·(σ_L/255)` | DCID — Dos Argenté |
| `Var(ΔI) > 500` | FTDD — Densité du Pelage |
| `C₃ = min(1, 2·ρ_contours + 0.5·R_d)` | FSGD — Morphologie Faciale |
| `S_E = 0.40·C₁ + 0.20·C₂ + 0.40·C₃` | Score Expert |
| `Decision = 0.60·ScoreIA + 0.40·S_E ≥ Γ=0.50` | Fusion décisionnelle |

---
**Auteur :** KANDOLO Herman · IS–VNU · hermankandolo2022@gmail.com

## 📦 Cellule 1 — Installation des dépendances

In [1]:
# Installation des bibliothèques nécessaires
!pip install -q ultralytics gradio opencv-python-headless Pillow

# Vérification GPU
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'✅ Device disponible : {device}')
if device == 'cuda':
    print(f'   GPU : {torch.cuda.get_device_name(0)}')
    print(f'   VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} Go')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 26.7 MB/s eta 0:00:00
✅ Device disponible : cuda
   GPU : Tesla T4
   VRAM : 15.6 Go


## 📂 Cellule 2 — Upload du modèle best_model.pt

**Deux options disponibles :**
- **Option A** : Upload manuel depuis votre PC
- **Option B** : Depuis Google Drive (recommandé)

In [2]:
import os

# ============================================================
# OPTION A — Upload direct depuis votre PC
# ============================================================
# Décommentez les lignes ci-dessous pour uploader manuellement

# from google.colab import files
# print('📤 Sélectionnez votre fichier best_model.pt ...')
# uploaded = files.upload()
# MODEL_PATH = list(uploaded.keys())[0]
# print(f'✅ Modèle uploadé : {MODEL_PATH}')


# ============================================================
# OPTION B — Depuis Google Drive (recommandé)
# ============================================================
# 1. Placez best_model.pt dans votre Google Drive
#    ex: Mon Drive/gorilla_detection/best_model.pt
# 2. Décommentez et exécutez :

from google.colab import drive
drive.mount('/content/drive')

# Adaptez ce chemin selon où vous avez mis le fichier dans Drive :
MODEL_PATH = '/content/drive/MyDrive/gorilla_detection/best_model.pt'

# Vérification
if os.path.exists(MODEL_PATH):
    size_mb = os.path.getsize(MODEL_PATH) / 1e6
    print(f'✅ Modèle trouvé : {MODEL_PATH}  ({size_mb:.1f} Mo)')
else:
    print(f'❌ Modèle introuvable : {MODEL_PATH}')
    print('   → Vérifiez le chemin ou utilisez l\'Option A (upload manuel)')

Mounted at /content/drive
✅ Modèle trouvé : /content/drive/MyDrive/gorilla_detection/best_model.pt  (5.5 Mo)


## 🧠 Cellule 3 — Code du Système Expert (MVH)

In [3]:
import cv2
import numpy as np
from ultralytics import YOLO
from PIL import Image
import json
from datetime import datetime
import traceback
import torch
import os

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# ═══════════════════════════════════════════════════════════════
# PARAMÈTRES DU SYSTÈME EXPERT (conformes article v12)
# ═══════════════════════════════════════════════════════════════
class ExpertKnowledge:
    SEUIL_C1   = 0.70   # DCID — Dos Argenté
    SEUIL_C3   = 0.65   # FSGD — Morphologie Faciale
    POIDS_C1   = 0.40
    POIDS_C2   = 0.20
    POIDS_C3   = 0.40
    THETA_CONF = 0.70   # seuil confiance YOLO
    THETA_VETO = 0.20   # seuil véto MVH
    GAMMA      = 0.50   # seuil décisionnel Zone 3
    ALPHA_IA   = 0.60
    ALPHA_MVH  = 0.40

    @staticmethod
    def calculer_SE(C1, C2_flag, C3):
        C2 = 1.0 if C2_flag else 0.0
        return round(ExpertKnowledge.POIDS_C1*C1 + ExpertKnowledge.POIDS_C2*C2 + ExpertKnowledge.POIDS_C3*C3, 4)

    @staticmethod
    def valider_classification(dl_classification, C1, C3, C2_flag, score_ia=0.0):
        if dl_classification == 'Aucun Objet Détecté':
            return 'Aucun Objet Détecté', 0.0, 0.0, 'N/A', 'Aucun gorille détecté.', []

        C2 = 1.0 if C2_flag else 0.0
        preuves = [
            {'nom':'C₁ — DCID (Dos Argenté)',       'zone':'ROI 20–60 %', 'score':C1,  'seuil':ExpertKnowledge.SEUIL_C1,      'poids':'40 %', 'contribution':round(ExpertKnowledge.POIDS_C1*C1,4),  'validee': C1 >= ExpertKnowledge.SEUIL_C1},
            {'nom':'C₂ — FTDD (Densité Pelage)',     'zone':'ROI entière', 'score':C2,  'seuil':'Var(ΔI) > 500',               'poids':'20 %', 'contribution':round(ExpertKnowledge.POIDS_C2*C2,4),  'validee': C2_flag},
            {'nom':'C₃ — FSGD (Morphologie Faciale)','zone':'ROI 0–40 %', 'score':C3,  'seuil':ExpertKnowledge.SEUIL_C3,      'poids':'40 %', 'contribution':round(ExpertKnowledge.POIDS_C3*C3,4),  'validee': C3 >= ExpertKnowledge.SEUIL_C3},
        ]

        S_E        = ExpertKnowledge.calculer_SE(C1, C2_flag, C3)
        score_ia_n = max(0.0, min(1.0, score_ia))

        if score_ia_n >= ExpertKnowledge.THETA_CONF:
            if S_E >= ExpertKnowledge.THETA_VETO:
                # Zone 1
                return dl_classification, S_E, round(score_ia_n, 4), 'Zone 1', (
                    f"✅ Zone 1 — YOLO certain + MVH confirme | "
                    f"ScoreIA={score_ia_n:.3f} ≥ {ExpertKnowledge.THETA_CONF} · "
                    f"S_E={S_E:.3f} ≥ θ_veto={ExpertKnowledge.THETA_VETO} → Décision YOLO acceptée"), preuves
            else:
                # Zone 2 — Véto
                statut = '⚠️ Véto MVH' if dl_classification == 'Gorille_Montagne' else '✅ Véto MVH (confirme)'
                return 'Autres_gorilles', S_E, round(score_ia_n, 4), 'Zone 2', (
                    f"{statut} | Zone 2 — ScoreIA={score_ia_n:.3f} ≥ {ExpertKnowledge.THETA_CONF} "
                    f"MAIS S_E={S_E:.3f} < θ_veto={ExpertKnowledge.THETA_VETO} → "
                    f"Absence traits morphologiques → Forcé : Autres_gorilles"), preuves
        else:
            # Zone 3 — Fusion
            decision = round(ExpertKnowledge.ALPHA_IA*score_ia_n + ExpertKnowledge.ALPHA_MVH*S_E, 4)
            classif  = 'Gorille_Montagne' if decision >= ExpertKnowledge.GAMMA else 'Autres_gorilles'
            if classif == dl_classification:
                statut = 'Validé ✅'
            elif classif == 'Gorille_Montagne':
                statut = 'Corrigé ↑ ✅'
            else:
                statut = 'Corrigé ↓'
            signe = '≥' if decision >= ExpertKnowledge.GAMMA else '<'
            return classif, S_E, decision, 'Zone 3', (
                f"{statut} | Zone 3 — YOLO incertain : ScoreIA={score_ia_n:.3f} < {ExpertKnowledge.THETA_CONF} → "
                f"Fusion : {ExpertKnowledge.ALPHA_IA}×{score_ia_n:.3f} + {ExpertKnowledge.ALPHA_MVH}×{S_E:.3f} "
                f"= {decision:.4f} {signe} Γ={ExpertKnowledge.GAMMA}"), preuves


# ═══════════════════════════════════════════════════════════════
# CHARGEMENT MODÈLE
# ═══════════════════════════════════════════════════════════════
_model_cache = {}

def charger_modele(model_path):
    if model_path in _model_cache:
        return _model_cache[model_path]
    if not os.path.exists(model_path):
        return None
    try:
        m = YOLO(model_path)
        m.to(DEVICE)
        dummy = np.zeros((640, 640, 3), dtype=np.uint8)
        m(dummy, conf=0.25, device=DEVICE, verbose=False)
        _model_cache[model_path] = m
        print(f'✅ Modèle chargé sur {DEVICE}')
        return m
    except Exception as e:
        print(f'❌ Erreur chargement : {e}')
        return None


# ═══════════════════════════════════════════════════════════════
# EXTRACTION DESCRIPTEURS MVH → clés C1, C3, C2_flag
# ═══════════════════════════════════════════════════════════════
def extraire_caracteristiques(image_array, bbox):
    try:
        x1, y1, x2, y2 = bbox
        h_img, w_img = image_array.shape[:2]
        x1 = max(0, min(x1, w_img-1)); y1 = max(0, min(y1, h_img-1))
        x2 = max(x1+1, min(x2, w_img)); y2 = max(y1+1, min(y2, h_img))

        roi = image_array[y1:y2, x1:x2]
        if roi.size == 0 or roi.shape[0] < 4 or roi.shape[1] < 4:
            return 0.0, 0.0, False
        if roi.dtype != np.uint8:
            roi = np.clip(roi, 0, 255).astype(np.uint8)
        if roi.ndim == 2:
            roi = cv2.cvtColor(roi, cv2.COLOR_GRAY2RGB)
        elif roi.shape[2] == 4:
            roi = cv2.cvtColor(roi, cv2.COLOR_RGBA2RGB)

        roi_gray = cv2.cvtColor(roi, cv2.COLOR_RGB2GRAY)
        height, _ = roi_gray.shape

        # C1 — DCID (zone 20–60 %)
        d1 = max(1, int(height*0.20)); d2 = max(d1+1, int(height*0.60))
        dorsal = roi_gray[d1:d2, :]
        if dorsal.size == 0:
            C1 = 0.0
        else:
            L_mean  = float(np.mean(dorsal))
            sigma_L = float(np.std(dorsal))
            R_b     = float(np.sum(dorsal > 150)) / dorsal.size
            C1 = min(1.0, (L_mean/255.0)*0.4 + R_b*0.4 + (sigma_L/255.0)*0.2)

        # C3 — FSGD (zone 0–40 %)
        f2 = max(1, int(height*0.40))
        facial = roi_gray[0:f2, :]
        if facial.size == 0:
            C3 = 0.0
        else:
            edges     = cv2.Canny(facial.astype(np.uint8), 50, 150)
            rho_edges = float(np.sum(edges > 0)) / edges.size
            R_d       = float(np.sum(facial < 100)) / facial.size
            C3 = min(1.0, rho_edges*2.0 + R_d*0.5)

        # C2 — FTDD (ROI entière)
        lap    = cv2.Laplacian(roi_gray.astype(np.float64), cv2.CV_64F)
        C2_flag = float(lap.var()) > 500.0

        return float(C1), float(C3), bool(C2_flag)

    except Exception as e:
        print(f'[extraire_caracteristiques] {e}')
        return 0.0, 0.0, False


# ═══════════════════════════════════════════════════════════════
# ANNOTATION IMAGE — classification FINALE (post-MVH)
# ═══════════════════════════════════════════════════════════════
def dessiner_detection(img, det, classif_finale, zone):
    if det['DL_Classification'] == 'Aucun Objet Détecté' or 'bbox' not in det:
        return img
    x1, y1, x2, y2 = det['bbox']
    h = y2-y1; w = x2-x1

    # Couleur BBox selon décision finale
    if zone == 'Zone 2' and det['DL_Classification'] == 'Gorille_Montagne':
        col = (255, 60, 60)    # rouge = véto sur détection positive
    elif classif_finale == 'Gorille_Montagne':
        col = (50, 205, 50)    # vert
    else:
        col = (255, 165, 0)    # orange

    cv2.rectangle(img, (x1,y1), (x2,y2), col, 3)

    # Zone C1 (vert/rouge)
    c1c = (0,220,0) if det['C1'] >= ExpertKnowledge.SEUIL_C1 else (255,80,80)
    cv2.rectangle(img, (x1+15, y1+int(h*0.20)), (x2-15, y1+int(h*0.60)), c1c, 2)
    cv2.putText(img, f"C1:{det['C1']:.2f}", (x1+18, y1+int(h*0.20)-8),
                cv2.FONT_HERSHEY_SIMPLEX, 0.52, c1c, 2)

    # Zone C3 (bleu/rouge)
    c3c = (80,80,255) if det['C3'] >= ExpertKnowledge.SEUIL_C3 else (255,80,80)
    cv2.rectangle(img, (x1+int(w*0.25), y1+8), (x2-int(w*0.25), y1+int(h*0.40)), c3c, 2)
    cv2.putText(img, f"C3:{det['C3']:.2f}", (x1+int(w*0.25)+5, y1+26),
                cv2.FONT_HERSHEY_SIMPLEX, 0.52, c3c, 2)

    # Label principal : classification FINALE + zone
    hdr_y1 = max(0, y1-65)
    if hdr_y1 == 0: hdr_y1 = y2+4
    label_w = max(300, w)
    cv2.rectangle(img, (x1, hdr_y1), (x1+label_w, hdr_y1+62), col, -1)
    cv2.putText(img, classif_finale, (x1+8, hdr_y1+22),
                cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255,255,255), 2)
    cv2.putText(img, f"{zone} | YOLO:{det['S0_general']:.0%} | S_E:{det['S_E']:.2f}",
                (x1+8, hdr_y1+46), cv2.FONT_HERSHEY_SIMPLEX, 0.48, (255,255,255), 1)
    return img


# ═══════════════════════════════════════════════════════════════
# PIPELINE COMPLET : YOLO → MVH → FUSION
# ═══════════════════════════════════════════════════════════════
def run_inference(image_pil, model_path):
    try:
        model = charger_modele(model_path)
        if model is None:
            return None, {'erreur': f'Modèle introuvable : {model_path}'}

        image_array = np.array(image_pil.convert('RGB'), dtype=np.uint8)
        h0, w0 = image_array.shape[:2]
        if max(h0, w0) > 1280:
            scale = 1280 / max(h0, w0)
            image_array = cv2.resize(image_array, (int(w0*scale), int(h0*scale)),
                                     interpolation=cv2.INTER_AREA)

        image_bgr = cv2.cvtColor(image_array, cv2.COLOR_RGB2BGR)
        results   = model(image_bgr, conf=0.25, device=DEVICE, verbose=False)

        if results and len(results[0].boxes) > 0:
            box       = results[0].boxes[0]
            x1,y1,x2,y2 = box.xyxy[0].cpu().numpy().astype(int)
            classe_id = int(box.cls[0].cpu().numpy())
            confiance = float(box.conf[0].cpu().numpy())
            class_names = {0:'Autres_gorilles', 1:'Gorille_Montagne'}
            dl_classif  = class_names.get(classe_id, 'Gorille_Montagne')

            C1, C3, C2_flag = extraire_caracteristiques(image_array, (x1,y1,x2,y2))
            S_E = ExpertKnowledge.calculer_SE(C1, C2_flag, C3)

            det = {
                'DL_Classification': dl_classif,
                'S0_general': confiance,
                'C1': C1, 'C3': C3, 'C2_flag': C2_flag,
                'S_E': S_E,
                'bbox': (int(x1),int(y1),int(x2),int(y2)),
                'classe_id': classe_id,
            }
        else:
            # Aucune détection — toutes les clés présentes avec valeurs neutres
            det = {
                'DL_Classification': 'Aucun Objet Détecté',
                'S0_general': 0.0,
                'C1': 0.0, 'C3': 0.0, 'C2_flag': False,
                'S_E': 0.0,
            }

        # Fusion AVANT annotation (annot affiche la décision finale)
        classif_finale, _, _, zone, _, _ = ExpertKnowledge.valider_classification(
            det['DL_Classification'], det['C1'], det['C3'],
            det['C2_flag'], score_ia=det['S0_general']
        )
        img_annotee = dessiner_detection(image_array.copy(), det, classif_finale, zone)
        return img_annotee, det

    except Exception as e:
        print(f'[run_inference] {e}\n{traceback.format_exc()}')
        return None, {'erreur': str(e)}


print('✅ Système Expert (MVH) chargé — clés C1/C3/C2_flag/S_E')
print(f'   θ_conf={ExpertKnowledge.THETA_CONF} | θ_veto={ExpertKnowledge.THETA_VETO} | Γ={ExpertKnowledge.GAMMA}')
print(f'   S_E = {ExpertKnowledge.POIDS_C1}·C₁ + {ExpertKnowledge.POIDS_C2}·C₂ + {ExpertKnowledge.POIDS_C3}·C₃')


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✅ Système Expert (MVH) chargé — clés C1/C3/C2_flag/S_E
   θ_conf=0.7 | θ_veto=0.2 | Γ=0.5
   S_E = 0.4·C₁ + 0.2·C₂ + 0.4·C₃


## 🚀 Cellule 4 — Lancement de l'interface Gradio

> Un **lien public** `https://xxxx.gradio.live` sera généré — valable **72h**, accessible depuis n'importe quel appareil.

In [ ]:
import gradio as gr

def analyser(image_pil, model_path):
    DASH = '—'
    if image_pil is None:
        return (None,
                DASH, DASH, DASH,
                DASH, DASH, DASH, DASH, DASH,
                DASH, DASH, DASH, DASH,
                '', '')

    image_annot, det = run_inference(image_pil, model_path)

    # ── Erreur run_inference ────────────────────────────────────
    if 'erreur' in det:
        msg = f"❌ {det['erreur']}"
        return (image_pil,
                msg, DASH, DASH,
                DASH, DASH, DASH, DASH, DASH,
                DASH, DASH, DASH, DASH,
                '', '')

    # ── Aucune détection YOLO ───────────────────────────────────
    if det['DL_Classification'] == 'Aucun Objet Détecté':
        aucun = '⚪ Aucun gorille détecté par YOLO'
        return (image_annot,
                aucun, '0.0 %', '⚪ Aucune détection',
                'N/A', DASH, DASH, DASH, DASH,
                '⚪ AUCUNE DÉTECTION', DASH, DASH, aucun,
                '', '')

    # ── Fusion MVH ──────────────────────────────────────────────
    classif_finale, S_E, decision, zone, explication, preuves = \
        ExpertKnowledge.valider_classification(
            det['DL_Classification'],
            det['C1'], det['C3'], det['C2_flag'],
            score_ia=det['S0_general']
        )

    # ── ÉTAPE 1 : YOLO ─────────────────────────────────────────
    yolo_badge = {'Gorille_Montagne':'🟢 Gorille_Montagne',
                  'Autres_gorilles': '🟡 Autres_gorilles'}.get(det['DL_Classification'], det['DL_Classification'])
    yolo_conf  = f"{det['S0_general']:.1%}"
    yolo_stat  = ('✅ CONFIANT' if det['S0_general'] >= ExpertKnowledge.THETA_CONF else '⚠️ INCERTAIN') + f" (θ_conf={ExpertKnowledge.THETA_CONF})"

    # ── ÉTAPE 2 : MVH ───────────────────────────────────────────
    c1_str = f"{det['C1']:.3f}  {'✅' if det['C1'] >= ExpertKnowledge.SEUIL_C1 else '❌'}  (seuil ≥ {ExpertKnowledge.SEUIL_C1})"
    c2_str = f"{'Oui ✅' if det['C2_flag'] else 'Non ❌'}  (Var. Laplacien > 500)"
    c3_str = f"{det['C3']:.3f}  {'✅' if det['C3'] >= ExpertKnowledge.SEUIL_C3 else '❌'}  (seuil ≥ {ExpertKnowledge.SEUIL_C3})"
    se_str = f"{S_E:.3f}  →  {'✅ ≥ θ_veto' if S_E >= ExpertKnowledge.THETA_VETO else '❌ < θ_veto'}  ({ExpertKnowledge.THETA_VETO})"

    # ── ÉTAPE 3 : Décision finale ───────────────────────────────
    final_badge = {'Gorille_Montagne':'🟢 GORILLE DE MONTAGNE',
                   'Autres_gorilles': '🟡 AUTRE GORILLE'}.get(classif_finale, '⚪ AUCUNE DÉTECTION')

    if classif_finale == det['DL_Classification']:
        delta = '✅ Confirmé par MVH'
    elif zone == 'Zone 2':
        delta = '⚠️ VÉTO MVH — YOLO contredit'
    else:
        delta = '🔄 Corrigé par fusion (Zone 3)'

    if zone == 'Zone 3':
        dec_str = (f"{ExpertKnowledge.ALPHA_IA}×{det['S0_general']:.3f} + "
                   f"{ExpertKnowledge.ALPHA_MVH}×{S_E:.3f} = {decision:.4f}  "
                   f"({'≥' if decision >= ExpertKnowledge.GAMMA else '<'} Γ={ExpertKnowledge.GAMMA})")
    elif zone == 'Zone 1':
        dec_str = f"YOLO direct — ScoreIA={det['S0_general']:.3f} (MVH confirme S_E={S_E:.3f})"
    else:
        dec_str = f"Véto MVH — S_E={S_E:.3f} < θ_veto={ExpertKnowledge.THETA_VETO}"

    # Tableau preuves
    tbl  = '| Descripteur | Zone ROI | Score | Seuil | Poids | Contrib. | ✓/✗ |\n'
    tbl += '|---|---|---|---|---|---|---|\n'
    for p in preuves:
        s = f"{p['score']:.3f}" if isinstance(p['score'], float) else str(p['score'])
        tbl += f"| {p['nom']} | {p['zone']} | {s} | {p['seuil']} | {p['poids']} | {p['contribution']:.3f} | {'✅' if p['validee'] else '❌'} |\n"
    tbl += f"| **S_E total** | — | **{S_E:.3f}** | θ_veto={ExpertKnowledge.THETA_VETO} | — | — | {'✅' if S_E >= ExpertKnowledge.THETA_VETO else '❌'} |\n"

    rapport = json.dumps({
        'date': datetime.now().isoformat(), 'seed': 42,
        'etape1_yolo':  {'classe': det['DL_Classification'], 'confiance': det['S0_general'], 'statut': yolo_stat},
        'etape2_mvh':   {'C1': det['C1'], 'C2_flag': det['C2_flag'], 'C3': det['C3'], 'S_E': S_E, 'zone': zone},
        'etape3_final': {'classification': classif_finale, 'decision': decision, 'delta': delta, 'explication': explication},
    }, indent=2, ensure_ascii=False)

    return (image_annot,
            yolo_badge, yolo_conf, yolo_stat,
            zone, c1_str, c2_str, c3_str, se_str,
            final_badge, delta, dec_str, explication,
            tbl, rapport)


# ── Interface Gradio ────────────────────────────────────────────
with gr.Blocks(title='Système Expert — Gorilles de Montagne') as demo:

    gr.Markdown("""
    # 🦍 Système Expert — Gorilles de Montagne
    ### YOLOv11n + Module de Validation Heuristique (MVH)
    > Résultats en **3 étapes** : 🤖 YOLO → 🧠 MVH → 🏆 Décision finale
    """)

    with gr.Row():
        model_input = gr.Textbox(label='📁 Chemin modèle (.pt)',
                                 value='/content/drive/MyDrive/gorilla_detection/best_model.pt', scale=4)
        btn = gr.Button('🚀 Analyser', variant='primary', scale=1)

    with gr.Row():
        img_in  = gr.Image(label='📷 Image d\'entrée', type='pil', height=400)
        img_out = gr.Image(label='🔍 Résultat annoté — décision FINALE · zones C₁(vert) C₃(bleu)', type='numpy', height=400)

    gr.Markdown('---\n### 🤖 Étape 1 — Ce que YOLO a décidé *(avant MVH)*')
    with gr.Row():
        e1_classe = gr.Textbox(label='Classe prédite par YOLO',        interactive=False, scale=2)
        e1_conf   = gr.Textbox(label='Score de confiance (ScoreIA)',    interactive=False, scale=1)
        e1_seuil  = gr.Textbox(label='Statut vis-à-vis θ_conf=0.70',   interactive=False, scale=2)

    gr.Markdown('---\n### 🧠 Étape 2 — Ce que le MVH a calculé')
    with gr.Row():
        e2_zone = gr.Textbox(label='🗺️ Zone décisionnelle',            interactive=False, scale=1)
        e2_c1   = gr.Textbox(label='C₁ — DCID (Dos Argenté)',          interactive=False, scale=2)
        e2_c2   = gr.Textbox(label='C₂ — FTDD (Texture Pelage)',       interactive=False, scale=2)
    with gr.Row():
        e2_c3   = gr.Textbox(label='C₃ — FSGD (Morphologie Faciale)',  interactive=False, scale=2)
        e2_se   = gr.Textbox(label='S_E = 0.40·C₁ + 0.20·C₂ + 0.40·C₃', interactive=False, scale=2)

    gr.Markdown('#### 🔬 Tableau des preuves MVH')
    preuves_out = gr.Markdown('*Uploadez une image et cliquez sur Analyser.*')

    gr.Markdown('---\n### 🏆 Étape 3 — Décision finale *(après fusion/véto)*')
    with gr.Row():
        e3_badge    = gr.Textbox(label='Classification finale',         interactive=False, scale=2)
        e3_delta    = gr.Textbox(label='Changement vs YOLO',            interactive=False, scale=2)
        e3_decision = gr.Textbox(label='Calcul de décision',            interactive=False, scale=3)
    e3_expl = gr.Textbox(label='💬 Explication complète', interactive=False, lines=2)

    with gr.Accordion('💾 Export JSON', open=False):
        json_out = gr.Code(label='Rapport JSON', language='json', lines=22)

    sorties = [img_out,
               e1_classe, e1_conf, e1_seuil,
               e2_zone, e2_c1, e2_c2, e2_c3, e2_se,
               e3_badge, e3_delta, e3_decision, e3_expl,
               preuves_out, json_out]

    btn.click(fn=analyser, inputs=[img_in, model_input], outputs=sorties)
    img_in.change(fn=analyser, inputs=[img_in, model_input], outputs=sorties)

demo.launch(share=True, debug=True)


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://a335ab29e374cea71b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


✅ Modèle chargé sur cuda


## 🖼️ Cellule 5 (optionnelle) — Test rapide sans interface

Utile pour tester une image directement dans Colab sans passer par Gradio.

In [ ]:
from google.colab import files
from IPython.display import display
import matplotlib.pyplot as plt

# Upload d'une image de test
print('📤 Sélectionnez une image de gorille ...')
uploaded = files.upload()
img_path = list(uploaded.keys())[0]

# Analyse
image_pil = Image.open(img_path).convert('RGB')
image_annot, det = run_inference(image_pil, MODEL_PATH)

classif, score_se, decision, explication, preuves = \
    ExpertKnowledge.valider_classification(
        det['DL_Classification'], det['S1_silverback'],
        det['S3_facial'], det['F2_pelage'], score_ia=det['S0_general']
    )

# Affichage
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
ax1.imshow(np.array(image_pil)); ax1.set_title('Image originale'); ax1.axis('off')
ax2.imshow(image_annot);         ax2.set_title('Détection annotée'); ax2.axis('off')
plt.tight_layout(); plt.show()

# Résumé
print('\n' + '='*60)
print(f'  Classification DL  : {det["DL_Classification"]} ({det["S0_general"]:.1%})')
print(f'  C1 Dos Argenté     : {det["S1_silverback"]:.3f}  (seuil ≥ 0.70)')
print(f'  C2 Densité Pelage  : {"Oui" if det["F2_pelage"] else "Non"}  (Var. Laplacien > 500)')
print(f'  C3 Morphologie     : {det["S3_facial"]:.3f}  (seuil ≥ 0.65)')
print(f'  S_E (Score Expert) : {score_se:.1f} %')
print(f'  Decision (fusion)  : {decision:.3f}  (Γ = 0.50)')
print('─'*60)
print(f'  ➤ RÉSULTAT FINAL   : {classif}')
print(f'  {explication}')
print('='*60)